In [1]:
import pandas as pd
import dask.dataframe as dd
import matplotlib.pyplot as plt
import seaborn as sns
import io
import gc

/usr/local/lib/python3.11/dist-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
nontraining = pd.read_csv('/content/drive/MyDrive/FlightFinalDescentAnomalyDetection/nontraining_flight_data.csv')
production = pd.read_csv('/content/drive/MyDrive/FlightFinalDescentAnomalyDetection/prod_flight_data.csv')
test = pd.read_csv('/content/drive/MyDrive/FlightFinalDescentAnomalyDetection/test_flight_data.csv')
training = pd.read_csv('/content/drive/MyDrive/FlightFinalDescentAnomalyDetection/training_flight_data.csv')
validation = pd.read_csv('/content/drive/MyDrive/FlightFinalDescentAnomalyDetection/valid_flight_data.csv')

In [4]:
training.columns = ['sample_id', 'time_step', 'aileron_pos_lh_deg', 'aileron_pos_rh_deg', 'corrected_angle_of_attack_deg', 'baro_correct_alt_lsp_ft', 'computed_airspeed_lsp_knots',
    'selected_course_deg', 'drift_angle_deg', 'elevator_pos_left_deg', 'te_flap_pos_disc', 'glideslope_dev_perc', 'selected_heading_deg', 'localizer_dev_perc', 'core_speed_avg_perc',
    'total_pressure_lsp_millibar', 'pitch_angle_lsp_deg', 'roll_angle_lsp_deg', 'rudder_pos_deg', 'true_heading_lsp_deg', 'vertical_accel_g', 'wind_speed_knots', 'label']

In [5]:
missing_count = training.isnull().sum()

print(training.isnull().sum())

sample_id                        0
time_step                        0
aileron_pos_lh_deg               0
aileron_pos_rh_deg               0
corrected_angle_of_attack_deg    0
baro_correct_alt_lsp_ft          0
computed_airspeed_lsp_knots      0
selected_course_deg              0
drift_angle_deg                  0
elevator_pos_left_deg            0
te_flap_pos_disc                 0
glideslope_dev_perc              0
selected_heading_deg             0
localizer_dev_perc               0
core_speed_avg_perc              0
total_pressure_lsp_millibar      0
pitch_angle_lsp_deg              0
roll_angle_lsp_deg               0
rudder_pos_deg                   0
true_heading_lsp_deg             0
vertical_accel_g                 0
wind_speed_knots                 0
label                            0
dtype: int64


In [6]:
float_columns = [col for col in training.columns if col not in ['sample_id', 'time_step', 'label'] and training[col].dtype == 'float64']

In [7]:
def text_eda(df):
    """Performs textual exploratory data analysis (EDA) on a DataFrame."""

    print("\n### Dataset Overview ###")
    print(f"Number of Rows: {df.shape[0]}")
    print(f"Number of Columns: {df.shape[1]}")
    print("Column Names:", list(df.columns))

    print("\n### Missing Values Analysis ###")
    missing_values = df.isnull().sum()
    print(missing_values[missing_values > 0])

    print("\n### Summary Statistics ###")
    print(df.describe())

    print("\n### Feature Types ###")
    print(df.dtypes.value_counts())

    print("\n### Sample Data ###")
    print(df.head())

In [8]:
df = training
text_eda(training)


### Dataset Overview ###
Number of Rows: 6389920
Number of Columns: 23
Column Names: ['sample_id', 'time_step', 'aileron_pos_lh_deg', 'aileron_pos_rh_deg', 'corrected_angle_of_attack_deg', 'baro_correct_alt_lsp_ft', 'computed_airspeed_lsp_knots', 'selected_course_deg', 'drift_angle_deg', 'elevator_pos_left_deg', 'te_flap_pos_disc', 'glideslope_dev_perc', 'selected_heading_deg', 'localizer_dev_perc', 'core_speed_avg_perc', 'total_pressure_lsp_millibar', 'pitch_angle_lsp_deg', 'roll_angle_lsp_deg', 'rudder_pos_deg', 'true_heading_lsp_deg', 'vertical_accel_g', 'wind_speed_knots', 'label']

### Missing Values Analysis ###
Series([], dtype: int64)

### Summary Statistics ###
          sample_id     time_step  aileron_pos_lh_deg  aileron_pos_rh_deg  \
count  6.389920e+06  6.389920e+06        6.389920e+06        6.389920e+06   
mean   4.985293e+04  7.950000e+01        8.414212e+01        8.252455e+01   
std    2.881442e+04  4.618712e+01        6.897701e+00        5.797395e+00   
min    0.000